<a href="https://colab.research.google.com/github/farrelrassya/LLM-Design-Patterns-Building-Robust-and-Efficient-AI-Systems/blob/main/04.Handling_Large_Datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bab 4 — Menangani Dataset Besar untuk Training LLM

*LLM Design Patterns* — notebook pendamping, revisi siap Google Colab (GPU)


## Masalahnya: data lebih besar dari mesin

Bayangkan kamu ingin melatih model bahasa dengan C4 (*Colossal Clean Crawled Corpus*), teks web hasil saringan Common Crawl yang dipakai melatih T5. Versi bahasa Inggrisnya sekitar 750 GB, jauh di atas RAM dan disk satu sesi Colab. Pertanyaan bab ini: bagaimana memproses data yang tidak bisa dimuat sekaligus?

Saat data membesar, ada lima titik yang tertekan:

- **storage**: data tidak muat di satu disk;
- **I/O**: membaca dari disk atau jaringan lebih lambat daripada kecepatan GPU memakan data;
- **preprocessing**: tokenisasi dan pembersihan teks dikerjakan per dokumen, dan untuk miliaran dokumen tahap ini bisa lebih lama dari training-nya;
- **memori**: dataset harus dialirkan atau dipotong-potong, bukan dimuat utuh;
- **kualitas**: makin besar data, makin sulit memastikan sampelnya tetap mewakili.

Notebook ini memakai potongan kecil C4 yang asli: 3.000 dokumen pertama dari satu shard. Ukuran ini cukup untuk mencoba dan mengukur setiap teknik dalam beberapa menit. Kodenya sama dengan yang dipakai pada skala besar; skalanya diatur lewat knob di sel konfigurasi.

### Bagian mana yang benar-benar memakai GPU?

Hampir semua teknik di bab ini adalah kerja **CPU dan I/O**: membaca file, mem-parse teks, menghitung hash, menulis Parquet. Operasi string tidak menjadi lebih cepat hanya karena ada GPU. GPU baru berguna ketika teks sudah menjadi tensor angka dan ada perhitungan matriks besar. Di notebook ini, itu terjadi saat model BERT mengubah token menjadi vektor embedding.

| Bagian | Pekerjaan utama | Perangkat |
|---|---|---|
| Streaming C4, sampling, Dask, sharding, format file, memmap | baca, parse, hash, tulis | CPU + disk/jaringan |
| Tokenisasi | memecah string menjadi ID token | CPU |
| Embedding BERT | perkalian matriks di encoder | **GPU** (CPU hanya bila diizinkan eksplisit) |

Karena itu notebook mengecek GPU satu kali di awal. Jika GPU tidak berfungsi, notebook berhenti dengan instruksi, bukan diam-diam pindah ke CPU.

**Catatan bukti.** File ini dikirim tanpa output, jadi semua angka di output adalah hasil run-mu sendiri. Angka di teks selalu diberi label sumbernya: *rumus* (misalnya peluang teoretis), *run historis* notebook asli (container CPU), atau *run validasi lokal* revisi ini (CPU di Mac, bukan Colab). Run validasi lokal tidak menjalankan CUDA dan tidak memakai bobot asli `bert-base-uncased`, jadi bagian GPU belum pernah diuji di Colab.

## Persiapan

Sel pertama memeriksa paket yang dibutuhkan dan hanya memasang yang belum ada. Di Colab biasanya tidak ada yang perlu dipasang, atau hanya `datasets` dan `dask`.

In [1]:
import subprocess
import sys

# modul yang di-import -> nama paket pip
PACKAGES = {
    "datasets": "datasets",
    "dask.dataframe": "dask[dataframe]",
    "dask.distributed": "distributed",
    "pyarrow": "pyarrow",
    "scipy": "scipy",
    "psutil": "psutil",
    "transformers": "transformers",
    "torch": "torch",
}

# Cek di proses terpisah agar kernel ini belum meng-import versi lama sebelum pemasangan.
probe = "import importlib, sys\nfor m in sys.argv[1:]:\n    try:\n        importlib.import_module(m)\n    except Exception:\n        print(m)"
result = subprocess.run([sys.executable, "-c", probe, *PACKAGES], capture_output=True, text=True)
missing = [PACKAGES[m] for m in result.stdout.split()]

if missing:
    print("memasang:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    print("Selesai. Jika sel berikutnya gagal karena konflik versi, pilih Runtime → Restart session "
          "lalu jalankan ulang dari atas.")
else:
    print("semua paket sudah tersedia")

semua paket sudah tersedia


Sel berikut memuat library dan mengumpulkan semua knob di satu tempat. Angkanya sengaja jauh lebih kecil dari buku, yang men-sample 100.000 dokumen dari C4 penuh. Kesimpulan di bawah berlaku untuk potongan ini. Untuk mencoba skala lebih besar, cukup ubah knob-nya.

In [2]:
import gc
import hashlib
import os
import platform
import sys
import time
from pathlib import Path

import datasets
import numpy as np
import pandas as pd
import psutil
import pyarrow
import torch
import transformers

# --- Perangkat ---
ALLOW_CPU = False        # True = sengaja jalan tanpa GPU, dengan profil embedding diperkecil

# --- Data: satu shard C4, bukan korpus penuh ---
C4_REPO = "allenai/c4"
C4_SHARD = "en/c4-train.00000-of-01024.json.gz"   # 1 dari 1024 shard train, ~319 MB terkompresi
N_DOCS = 3000            # dokumen yang diambil dari awal shard itu
NUM_SAMPLES = 305        # target sampling (buku: 100000); sengaja tidak habis dibagi 10

# --- Model untuk tokenisasi dan embedding (tokenizer sama dengan buku) ---
MODEL_NAME = "bert-base-uncased"
EMBED_PROFILES = {
    "gpu": {"docs": N_DOCS, "max_length": 256, "batch_size": 64},
    "cpu_reduced": {"docs": 192, "max_length": 128, "batch_size": 16},
}

# --- Eksperimen I/O ---
MEMMAP_TEST_MB = 200     # ukuran array untuk uji memmap vs full load
TIMING_REPEATS = 5       # setiap pengukuran waktu diambil mediannya

WORK_DIR = Path("ch04_work")
WORK_DIR.mkdir(exist_ok=True)

print("python      :", sys.version.split()[0], platform.machine())
print("torch       :", torch.__version__, "| CUDA build:", torch.version.cuda)
print("transformers:", transformers.__version__, "| datasets:", datasets.__version__, "| pyarrow:", pyarrow.__version__)
print("CPU logis   :", os.cpu_count(), "| RAM:", round(psutil.virtual_memory().total / 2**30, 1), "GB")

python      : 3.13.16 x86_64
torch       : 2.11.0+cu130 | CUDA build: 13.0
transformers: 5.20.2 | datasets: 4.8.5 | pyarrow: 23.0.1
CPU logis   : 2 | RAM: 12.7 GB


### Cek GPU fungsional

`torch.cuda.is_available()` saja belum cukup: driver bisa terdeteksi, tetapi alokasi atau komputasinya gagal. Cek di bawah benar-benar membuat dua matriks di GPU, mengalikannya, menunggu GPU selesai (`synchronize`), memastikan tidak ada NaN/Inf, lalu membandingkan hasilnya dengan perhitungan CPU.

Cek ini juga memilih presisi untuk bagian embedding. GPU generasi Ampere ke atas (compute capability ≥ 8) mendukung bfloat16. T4 di Colab gratis berada di generasi 7.5 dan tidak mendukungnya, sehingga T4 memakai float16.

In [3]:
def cuda_functional_check():
    device = torch.device("cuda:0")
    a = torch.randn(512, 512, device=device)
    b = torch.randn(512, 512, device=device)
    c = a @ b
    torch.cuda.synchronize(device)
    if not torch.isfinite(c).all().item():
        raise RuntimeError("hasil matmul GPU mengandung NaN/Inf")
    reference = a.double().cpu() @ b.double().cpu()
    rel_err = ((c.double().cpu() - reference).abs().max() / reference.abs().max()).item()
    if rel_err > 1e-2:
        raise RuntimeError(f"matmul GPU menyimpang dari CPU (error relatif {rel_err:.1e})")
    return device, rel_err


if torch.cuda.is_available():
    DEVICE, gate_err = cuda_functional_check()     # gagal = berhenti di sini, tanpa fallback
    props = torch.cuda.get_device_properties(DEVICE)
    free_bytes, total_bytes = torch.cuda.mem_get_info(DEVICE)
    AMP_DTYPE = torch.bfloat16 if props.major >= 8 else torch.float16
    PROFILE_NAME = "gpu"
    print(f"GPU OK: {props.name} | compute capability {props.major}.{props.minor} | "
          f"memori bebas {free_bytes / 2**30:.1f} / {total_bytes / 2**30:.1f} GB | "
          f"error relatif matmul vs CPU {gate_err:.1e}")
elif ALLOW_CPU:
    DEVICE = torch.device("cpu")
    AMP_DTYPE = None
    PROFILE_NAME = "cpu_reduced"
    print("PERINGATAN: berjalan di CPU karena ALLOW_CPU=True; embedding memakai profil diperkecil.")
else:
    raise RuntimeError(
        "GPU CUDA tidak terdeteksi. Di Colab: Runtime → Change runtime type → pilih GPU (mis. T4), "
        "lalu jalankan ulang dari atas. Untuk sengaja memakai CPU dengan profil diperkecil, "
        "set ALLOW_CPU = True di sel konfigurasi."
    )

PROFILE = EMBED_PROFILES[PROFILE_NAME]
print("perangkat:", DEVICE, "| presisi autocast:", AMP_DTYPE, "| profil embedding:", PROFILE_NAME, PROFILE)

GPU OK: Tesla T4 | compute capability 7.5 | memori bebas 14.4 / 14.6 GB | error relatif matmul vs CPU 9.3e-07
perangkat: cuda:0 | presisi autocast: torch.float16 | profil embedding: gpu {'docs': 3000, 'max_length': 256, 'batch_size': 64}


## 1. Streaming: membaca korpus tanpa mengunduhnya

Cara paling naif memakai C4 adalah mengunduh semuanya lalu membacanya. Untuk 750 GB, itu mustahil di Colab. **Streaming** membalik urutannya: dataset dibaca sebagai aliran record dari file remote, dan record berikutnya baru diambil ketika loop memintanya. Yang menyentuh disk lokal hanya metadata kecil.

Buku memakai library `datasets` dari Hugging Face untuk ini, lalu menjalankan fungsi preprocessing secara paralel di beberapa core.

> **Listing buku: memuat C4 lalu `map` paralel — perlu diperbaiki**
>
> ```python
> dataset = load_dataset(dataset_name, streaming=True)   # dataset_name = "c4"
> processed_dataset = dataset.map(
>     preprocess_function, batched=True, num_proc=num_proc,
>     remove_columns=dataset["train"].column_names)
> ```
>
> Ada dua masalah, dan keduanya juga muncul lagi di validasi lokal (`datasets` 5.1.0). Nama `"c4"` gagal dengan `RuntimeError: Dataset scripts are no longer supported, but found c4.py`, karena C4 kini disimpan sebagai repo data `allenai/c4`. Setelah namanya diperbaiki, `map` pada dataset streaming menolak argumen `num_proc` (`TypeError: ... unexpected keyword argument 'num_proc'`). Versi yang jalan ada di sel berikut, dan `num_proc` dipindah ke tempat yang benar sesudahnya.

In [4]:
from datasets import load_dataset


def open_c4_stream():
    # Aliran record dari SATU shard C4; tidak ada file yang diunduh penuh.
    return load_dataset(C4_REPO, data_files={"train": C4_SHARD}, split="train", streaming=True)


def normalize_whitespace(batch):
    batch["text"] = [" ".join(text.split()) for text in batch["text"]]
    return batch


stream = open_c4_stream().map(normalize_whitespace, batched=True)
print(stream)

first_five_urls = []
for example in stream.take(5):
    first_five_urls.append(example["url"])
    print(example["url"][:70])
    print("   ", example["text"][:90], "...")

README.md:   0%|          | 0.00/41.1k [00:00<?, ?B/s]

IterableDataset({
    features: Unknown,
    num_shards: 1
})
https://klyq.com/beginners-bbq-class-taking-place-in-missoula/
    Beginners BBQ Class Taking Place in Missoula! Do you want to get better at making deliciou ...
https://forums.macrumors.com/threads/restore-from-larger-disk-to-small
    Discussion in 'Mac OS X Lion (10.7)' started by axboi87, Jan 20, 2012. I've got a 500gb in ...
https://awishcometrue.com/Catalogs/Clearance/Tweens/V1960-Find-A-Way
    Foil plaid lycra and spandex shortall with metallic slinky insets. Attached metallic elast ...
https://www.blackhatworld.com/seo/how-many-backlinks-per-day-for-new-s
    How many backlinks per day for new site? Discussion in 'Black Hat SEO' started by Omoplata ...
http://bond.dpsk12.org/category/news/
    The Denver Board of Education opened the 2017-18 school year with an update on projects th ...


Lima dokumen pertama muncul tanpa menunggu shard 319 MB itu terunduh. Isinya khas web crawl: kelas memasak, forum, toko online. Ada tiga hal yang terjadi di sini:

- `data_files` membatasi sumber ke satu shard, jadi data yang mungkin tersentuh punya batas atas yang jelas.
- `map(..., batched=True)` belum menjalankan apa pun saat dipanggil. Fungsi normalisasi baru dieksekusi per batch ketika loop mengambil record. Pola ini disebut *lazy*.
- Setiap pemanggilan `take(5)` membuat iterasi baru dari awal shard. Iterator Python biasa justru habis setelah dibaca sekali. Perbedaan kecil ini menentukan apakah dua tahap pipeline melihat dokumen yang sama.

Lalu ke mana `num_proc` dari buku? `num_proc` membagi dataset ke beberapa proses berdasarkan posisi baris, jadi ia butuh dataset yang panjang dan posisinya diketahui. Aliran tidak punya keduanya. Tempat yang tepat untuk `num_proc` adalah dataset yang sudah dimaterialisasi, seperti korpus kerja di bawah.

### Korpus kerja: 3.000 dokumen nyata

Bagian sampling, sharding, dan format file butuh data yang bisa dipegang utuh. Kita ambil `N_DOCS` dokumen pertama dari shard yang sama dan menyimpannya sebagai list of dict `{"id", "text"}`. C4 tidak punya kolom `id`, jadi URL dokumen dipakai sebagai `id`, karena listing sharding buku membutuhkannya.

In [5]:
t0 = time.perf_counter()
corpus = [{"id": ex["url"], "text": ex["text"]} for ex in open_c4_stream().take(N_DOCS)]
pull_seconds = time.perf_counter() - t0

assert len(corpus) == N_DOCS, f"stream berhenti di {len(corpus)} dokumen"
assert [d["id"] for d in corpus[:5]] == first_five_urls, "iterasi baru harus mulai dari awal shard"

lengths = np.array([len(d["text"]) for d in corpus])
print(f"{len(corpus)} dokumen dalam {pull_seconds:.1f} s | id unik: {len({d['id'] for d in corpus})}")
print(f"panjang (karakter): min={lengths.min()}  median={np.median(lengths):.0f}  "
      f"mean={lengths.mean():.0f}  max={lengths.max()}")
print(f"total teks: {lengths.sum() / 1e6:.1f} juta karakter")

3000 dokumen dalam 3.3 s | id unik: 3000
panjang (karakter): min=37  median=1214  mean=2216  max=69815
total teks: 6.6 juta karakter


Assert kedua memastikan korpus dimulai dari lima dokumen yang sama dengan yang dicetak di atas. Artinya, setiap tahap memang membaca shard dari awal.

Perhatikan median dan mean panjang dokumen. Pada run validasi lokal (shard yang sama), median 1.214 karakter, mean 2.216, dan dokumen terpanjang 69.815 karakter. Setengah dokumen lebih pendek dari kira-kira 200 kata, tetapi segelintir dokumen yang sangat panjang menarik rata-rata hampir dua kali lipat median. Distribusi yang miring ke kanan seperti ini membuat *stratified sampling* di bagian 3 relevan.

Sekarang `num_proc` punya tempat yang benar. Korpus dimuat menjadi `Dataset` biasa (non-streaming), lalu `map` dibagi ke beberapa proses. Hasilnya harus identik dengan versi satu proses; yang berubah hanya cara kerjanya dibagi.

In [6]:
from datasets import Dataset

corpus_ds = Dataset.from_list(corpus)


def lowercase_batch(batch):
    return {"text_lower": [text.lower() for text in batch["text"]]}


n_proc = max(1, min(2, os.cpu_count() or 1))
parallel_ds = corpus_ds.map(lowercase_batch, batched=True, num_proc=n_proc)
serial_ds = corpus_ds.map(lowercase_batch, batched=True)

assert list(parallel_ds["text_lower"]) == list(serial_ds["text_lower"])
print(f"num_proc={n_proc}: {parallel_ds.num_rows} baris, hasil identik dengan versi 1 proses")

Map (num_proc=2):   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

num_proc=2: 3000 baris, hasil identik dengan versi 1 proses


## 2. Dari teks ke tensor: tokenisasi, DataLoader, lalu GPU

Model tidak membaca string. Teks harus dipecah menjadi **token** (potongan kata) dan setiap token diganti ID angka. Itu tugas **tokenizer**, dan pekerjaannya murni manipulasi string di CPU. Supaya beberapa dokumen bisa diproses sekaligus sebagai satu tensor, panjangnya disamakan dengan **padding** (diisi token kosong) dan **truncation** (dipotong jika terlalu panjang). Barulah tensor itu dipindah ke GPU untuk perhitungan model.

Buku membungkus semua langkah ini dalam satu `DataLoader` PyTorch.

> **Listing buku: tokenisasi di dalam `DataLoader` — perlu diperbaiki**
>
> ```python
> def process_batch(batch):
>     return {k: v.to(device) for k, v in preprocess(batch).items()}
>
> DataLoader(dataset["train"].map(process_batch), batch_size=batch_size,
>            num_workers=2, pin_memory=True)
> ```
>
> Akar masalahnya ada di `map`. Tanpa `batched=True`, `map` memproses satu dokumen per panggilan, lalu `datasets` menyimpan kembali tensor dari tokenizer sebagai list Python. Kolom asli (`text`, `url`, `timestamp`) juga ikut terbawa ke `DataLoader`. Gejalanya bergantung versi. Pada run historis (`datasets` 5.0.1), loop pencetak shape berhenti dengan `AttributeError: 'list' object has no attribute 'shape'`. Pada validasi lokal (`datasets` 5.1.0), kegagalannya lebih awal: `TypeError: default_collate: ... found <class 'datetime.datetime'>`, karena kolom `timestamp` kini dibaca sebagai datetime. Memindahkan tensor ke `device` di dalam `map` juga menaruh transfer GPU di tahap yang salah. Versi yang jalan: tokenisasi batched → buang kolom teks → format torch → `DataLoader` → tensor dipindah ke GPU di loop model.

In [7]:
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_batch(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=512)


def make_token_loader(batch_size, n_docs):
    tokenized = (
        open_c4_stream()
        .take(n_docs)
        .map(tokenize_batch, batched=True, batch_size=batch_size,
             remove_columns=["text", "timestamp", "url"])
        .with_format("torch")
    )
    return DataLoader(tokenized, batch_size=batch_size, num_workers=0)


for i, batch in enumerate(make_token_loader(batch_size=32, n_docs=5 * 32)):
    shapes = {k: tuple(v.shape) for k, v in batch.items()}
    print(f"batch {i}: {shapes} | {batch['input_ids'].dtype} | device {batch['input_ids'].device}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

batch 0: {'input_ids': (32, 512), 'token_type_ids': (32, 512), 'attention_mask': (32, 512)} | torch.int64 | device cpu
batch 1: {'input_ids': (32, 512), 'token_type_ids': (32, 512), 'attention_mask': (32, 512)} | torch.int64 | device cpu
batch 2: {'input_ids': (32, 512), 'token_type_ids': (32, 512), 'attention_mask': (32, 512)} | torch.int64 | device cpu
batch 3: {'input_ids': (32, 512), 'token_type_ids': (32, 512), 'attention_mask': (32, 512)} | torch.int64 | device cpu
batch 4: {'input_ids': (32, 512), 'token_type_ids': (32, 512), 'attention_mask': (32, 512)} | torch.int64 | device cpu


Lima batch berbentuk `(32, 512)`: 32 dokumen, masing-masing 512 token. Tensornya masih di CPU, dan memang seharusnya begitu, karena tokenisasi adalah kerja CPU. Tensor dipindah ke GPU nanti, tepat sebelum dipakai model.

`padding="max_length"` membuat setiap dokumen sepanjang 512 token. Pertanyaan berikutnya: berapa banyak dari 512 posisi itu yang hanya padding? Ini penting karena GPU tetap menghitung setiap posisi, termasuk yang kosong.

In [8]:
sample_texts = [d["text"] for d in corpus[:1024]]
token_lengths = np.array([len(ids) for ids in tokenizer(sample_texts, truncation=True, max_length=512)["input_ids"]])


def padding_fraction(lengths, batch_size, pad_to=None):
    padded_total = 0
    for start in range(0, len(lengths), batch_size):
        chunk = lengths[start:start + batch_size]
        width = pad_to if pad_to is not None else chunk.max()
        padded_total += width * len(chunk)
    return 1 - lengths.sum() / padded_total


print(f"token per dokumen (dipotong di 512): median {np.median(token_lengths):.0f} | "
      f"{(token_lengths == 512).mean():.0%} dokumen terpotong")
print(f"porsi padding, max_length=512           : {padding_fraction(token_lengths, 32, pad_to=512):.1%}")
print(f"porsi padding, padding dinamis          : {padding_fraction(token_lengths, 32):.1%}")
print(f"porsi padding, dinamis + urut panjang   : {padding_fraction(np.sort(token_lengths), 32):.1%}")

token per dokumen (dipotong di 512): median 260 | 26% dokumen terpotong
porsi padding, max_length=512           : 43.9%
porsi padding, padding dinamis          : 43.9%
porsi padding, dinamis + urut panjang   : 2.5%


Porsi padding adalah persentase posisi tensor yang tidak berisi teks. Pada run validasi lokal (tokenizer `bert-base-uncased` asli, 1.024 dokumen pertama), median panjang dokumen 260 token dan 26% dokumen terpotong di 512. Dengan `max_length=512`, 43,9% posisi yang dihitung GPU hanyalah padding.

- **Padding dinamis** memakai lebar dokumen terpanjang di setiap batch. Di data ini cara tersebut tidak menolong sama sekali (tetap 43,9%): karena seperempat dokumen terpotong di 512, hampir setiap batch berisi 32 dokumen acak yang memuat minimal satu dokumen 512 token, sehingga lebar batch tetap 512.
- **Bucketing** mengurutkan dokumen menurut panjang sebelum dibagi ke batch, sehingga dokumen dalam satu batch mirip panjangnya. Porsi padding turun ke 2,5%.

Angka ini berasal dari data dan tokenizer yang sama dengan yang akan kamu pakai, jadi output-mu seharusnya identik. Bucketing adalah cara yang dipakai saat menghitung embedding di bawah.

### `num_workers` dan bahaya data ganda

Buku memakai `num_workers=2` agar tokenisasi berjalan di proses lain. Untuk dataset yang panjangnya diketahui (*map-style*), PyTorch membagi indeks ke setiap worker. Untuk **IterableDataset** (aliran), PyTorch tidak tahu cara membaginya: setiap worker menerima salinan dataset yang sama dan menjalankan iteratornya sendiri. Jika dataset tidak membagi dirinya per worker, setiap record muncul sekali per worker.

Contoh kecil: aliran 12 record dan `DataLoader` dengan 2 worker. Kelasnya disimpan di modul kecil karena worker adalah proses terpisah yang harus bisa meng-import kode itu.

In [9]:
helper_dir = str(WORK_DIR.resolve())
if helper_dir not in sys.path:
    sys.path.insert(0, helper_dir)

(WORK_DIR / "ch04_stream_workers.py").write_text('''
from torch.utils.data import IterableDataset, get_worker_info


class NaiveStream(IterableDataset):
    # Setiap worker mengulang seluruh aliran.
    def __init__(self, n):
        self.n = n

    def __iter__(self):
        return iter(range(self.n))


class ShardedStream(NaiveStream):
    # Worker ke-k hanya mengambil record k, k+W, k+2W, ...
    def __iter__(self):
        info = get_worker_info()
        worker_id, num_workers = (0, 1) if info is None else (info.id, info.num_workers)
        return iter(range(worker_id, self.n, num_workers))
''')

from ch04_stream_workers import NaiveStream, ShardedStream


def collect(dataset, num_workers):
    loader = DataLoader(dataset, batch_size=None, num_workers=num_workers)
    return sorted(int(x) for x in loader)


naive = collect(NaiveStream(12), num_workers=2)
sharded = collect(ShardedStream(12), num_workers=2)
print(f"naif   : {len(naive)} record, {len(naive) - len(set(naive))} duplikat")
print(f"dibagi : {len(sharded)} record, {len(sharded) - len(set(sharded))} duplikat")
assert sharded == list(range(12))

naif   : 24 record, 12 duplikat
dibagi : 12 record, 0 duplikat


Dataset naif mengubah 12 record menjadi 24 karena setiap worker mengirim seluruh aliran. Model akan melihat setiap dokumen dua kali tanpa error apa pun. Versi yang membagi diri lewat `get_worker_info()` menghasilkan tepat 12.

`IterableDataset` dari Hugging Face sudah melakukan pembagian ini per shard file. Aliran kita hanya punya satu shard (`num_shards: 1` pada output streaming di atas). Dengan `num_workers=2`, satu worker kebagian shard itu, sementara worker lain dihentikan tanpa data; `datasets` mencatatnya sebagai peringatan "Stopping 1 dataloader workers". Pada validasi lokal, 64 record pertama lewat 2 worker tetap unik dan urutannya sama dengan `take(64)`. Jadi tidak ada duplikasi, tetapi juga tidak ada percepatan. Karena itu loader tokenisasi di atas memakai `num_workers=0`.

### Bagian GPU: menghitung embedding dengan BERT

Di sinilah GPU benar-benar bekerja. Encoder BERT mengubah setiap token menjadi vektor 768 angka lewat 12 lapis perkalian matriks. Vektor token setiap dokumen lalu dirata-rata, hanya pada posisi yang bukan padding (*mean pooling*), sehingga satu dokumen diwakili satu **embedding** berdimensi 768. Embedding ini dipakai lagi di bagian format file dan memmap, jadi pipeline-nya utuh: teks → token → vektor → file.

Kodenya mencatat dua waktu terpisah supaya jelas siapa mengerjakan apa: tokenisasi (CPU) serta transfer + forward pass (perangkat model). Setiap batch ditunggu sampai selesai (`synchronize`) agar waktu GPU tidak tercatat terlalu kecil. Dokumen diurutkan menurut panjang sebelum dibagi ke batch, lalu hasilnya dikembalikan ke urutan semula.

Jika saat memuat model muncul laporan bahwa bobot `cls.*` tidak dipakai, itu wajar: checkpoint `bert-base-uncased` juga berisi kepala untuk pretraining, sedangkan kita hanya butuh encoder-nya.

In [10]:
from transformers import AutoModel

model = AutoModel.from_pretrained(MODEL_NAME)
model = model.to(DEVICE).eval()
assert next(model.parameters()).device == DEVICE
n_params = sum(p.numel() for p in model.parameters())
print(f"{MODEL_NAME}: {n_params / 1e6:.0f} juta parameter di {DEVICE} | hidden size {model.config.hidden_size}")

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


bert-base-uncased: 109 juta parameter di cuda:0 | hidden size 768


In [11]:
def embed_texts(texts, batch_size, max_length):
    # Kembalikan (embedding float32 di CPU, detik tokenisasi, detik transfer + model).
    order = np.argsort([len(t) for t in texts])          # bucketing berdasarkan panjang
    out = np.zeros((len(texts), model.config.hidden_size), dtype=np.float32)
    filled = np.zeros(len(texts), dtype=bool)
    tok_seconds = 0.0
    model_seconds = 0.0
    for start in range(0, len(texts), batch_size):
        idx = order[start:start + batch_size]

        t0 = time.perf_counter()
        enc = tokenizer([texts[i] for i in idx], padding=True, truncation=True,
                        max_length=max_length, return_tensors="pt")
        tok_seconds += time.perf_counter() - t0

        t0 = time.perf_counter()
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.inference_mode(), torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            hidden = model(**enc).last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1).float()
        pooled = (hidden.float() * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize(DEVICE)
        model_seconds += time.perf_counter() - t0

        assert pooled.device == enc["input_ids"].device == next(model.parameters()).device
        if not torch.isfinite(pooled).all().item():
            raise RuntimeError(f"embedding non-finite pada batch mulai {start}")
        out[idx] = pooled.cpu().numpy()
        filled[idx] = True
    assert filled.all()
    return out, tok_seconds, model_seconds


embed_input = [d["text"] for d in corpus[:PROFILE["docs"]]]
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats(DEVICE)

doc_emb, tok_s, model_s = embed_texts(embed_input, PROFILE["batch_size"], PROFILE["max_length"])
n_emb = len(doc_emb)

print(f"embedding: {doc_emb.shape} {doc_emb.dtype} | dihitung di {DEVICE}")
print(f"tokenisasi (CPU)            : {tok_s:6.2f} s")
print(f"transfer + model ({DEVICE.type:>4})     : {model_s:6.2f} s -> {n_emb / model_s:,.0f} dokumen/s")
if DEVICE.type == "cuda":
    print(f"puncak memori GPU teralokasi: {torch.cuda.max_memory_allocated(DEVICE) / 2**20:,.0f} MB")

embedding: (3000, 768) float32 | dihitung di cuda:0
tokenisasi (CPU)            :   4.63 s
transfer + model (cuda)     :   7.73 s -> 388 dokumen/s
puncak memori GPU teralokasi: 840 MB


Baca dua baris waktu itu bersama-sama. Jika tokenisasi CPU sama lama atau lebih lama dari forward pass, GPU menganggur menunggu data. Itulah masalah I/O dan preprocessing dari awal bab, kini terlihat dalam angka. Ada dua solusi lazim: tokenisasi dijalankan sekali dan hasilnya disimpan (bagian 6), atau tokenisasi dijalankan di worker `DataLoader` sementara GPU menghitung batch sebelumnya. Angka dokumen/detik hanya berlaku untuk profil ini, yaitu `max_length`, ukuran batch, dan GPU yang kamu pakai.

Presisi campuran (float16 di T4) mempercepat GPU, tetapi perlu dicek apakah hasilnya masih sama dengan perhitungan float32. Kontrolnya: empat dokumen pertama dihitung ulang dengan salinan model float32 di CPU, lalu arah kedua vektor dibandingkan dengan *cosine similarity* (1,0 berarti arahnya identik).

In [12]:
if DEVICE.type == "cuda":
    cpu_model = AutoModel.from_pretrained(MODEL_NAME).eval()
    enc = tokenizer(embed_input[:4], padding=True, truncation=True,
                    max_length=PROFILE["max_length"], return_tensors="pt")
    with torch.inference_mode():
        hidden = cpu_model(**enc).last_hidden_state
    mask = enc["attention_mask"].unsqueeze(-1).float()
    reference = ((hidden * mask).sum(dim=1) / mask.sum(dim=1)).numpy()
    cos = (reference * doc_emb[:4]).sum(axis=1) / (
        np.linalg.norm(reference, axis=1) * np.linalg.norm(doc_emb[:4], axis=1))
    print("cosine embedding GPU-autocast vs CPU-float32:", np.round(cos, 5))
    assert (cos > 0.999).all(), "embedding GPU menyimpang dari referensi float32"
    del cpu_model
else:
    print("Dilewati: perangkat saat ini sudah CPU float32, jadi tidak ada presisi campuran untuk dicek.")

del model
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    print(f"memori GPU teralokasi setelah model dilepas: {torch.cuda.memory_allocated(DEVICE) / 2**20:.0f} MB")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


cosine embedding GPU-autocast vs CPU-float32: [1. 1. 1. 1.]
memori GPU teralokasi setelah model dilepas: 9 MB


## 3. Sampling: memperkecil data tanpa mengubah wajahnya

Jika 3.000 dokumen terlalu banyak untuk sebuah eksperimen, kita ambil sebagian. Risikonya, sampel bisa kebetulan berisi terlalu banyak dokumen pendek, sehingga model yang dilatih dengannya melihat data yang berbeda dari populasinya. Buku menyebut empat cara mengambil sampel:

- **random sampling**: setiap dokumen berpeluang sama. Caranya sederhana, tetapi kelompok kecil bisa terlewat.
- **systematic sampling**: ambil setiap dokumen ke-k setelah titik awal acak. Cara ini berbahaya bila urutan data punya pola berulang.
- **reservoir sampling**: menjaga sampel berukuran tetap dari aliran yang panjangnya tidak diketahui, cocok untuk streaming.
- **stratified sampling**: data dibagi ke kelompok (**strata**) menurut atribut penting, lalu setiap kelompok diambil sesuai porsinya.

Buku membuat strata dari **panjang teks**, karena panjang input memengaruhi perilaku training. Batas strata berupa persentil: dengan 10 strata, stratum pertama berisi 10% dokumen terpendek, stratum kedua 10% berikutnya, dan seterusnya.

> **Listing buku: `stratified_length_sampling` — berjalan, tetapi hasilnya bisa menyesatkan**
>
> ```python
> strata_bounds = np.percentile(lengths, np.linspace(0, 100, num_strata + 1))
> stratum = [example for example in dataset
>            if strata_bounds[i] <= len(example['text']) < strata_bounds[i+1]]
> stratum_samples = np.random.choice(stratum, size=num_samples // num_strata, replace=False)
> ```
>
> Listing ini tidak memunculkan error, tetapi punya dua cacat. Pertama, batas atas memakai `<` untuk semua stratum, padahal batas atas stratum terakhir adalah panjang maksimum, sehingga dokumen terpanjang tidak masuk stratum mana pun. Kedua, `num_samples // num_strata` membulatkan ke bawah: permintaan 305 sampel hanya menghasilkan 300. Jika banyak dokumen sama panjang, sebuah stratum bisa kosong, dan `replace=False` lalu memicu `ValueError` (dicek pada data kontrol sintetis). Sel berikut mengukur cacat pertama pada korpus kita.

In [13]:
book_bounds = np.percentile(lengths, np.linspace(0, 100, 11))
in_some_stratum = np.zeros(len(lengths), dtype=bool)
for i in range(10):
    in_some_stratum |= (book_bounds[i] <= lengths) & (lengths < book_bounds[i + 1])

print("dokumen di luar semua strata (aturan buku):", int((~in_some_stratum).sum()),
      "| panjangnya:", sorted(lengths[~in_some_stratum].tolist()))
print(f"meminta {NUM_SAMPLES} sampel -> buku mengambil {NUM_SAMPLES // 10} per stratum = {NUM_SAMPLES // 10 * 10} sampel")

dokumen di luar semua strata (aturan buku): 1 | panjangnya: [69815]
meminta 305 sampel -> buku mengambil 30 per stratum = 300 sampel


Dokumen yang terlewat adalah dokumen terpanjang. Panjangnya sama dengan persentil ke-100, sedangkan aturan `<` menolaknya. Pada korpus yang miring ke kanan, dokumen di ekor panjang justru yang paling langka.

Versi perbaikan di bawah memakai tiga ide:

- setiap dokumen mendapat tepat satu nomor stratum lewat `np.searchsorted`, termasuk dokumen terpanjang;
- kuota setiap stratum sebanding dengan ukurannya, sehingga stratum yang membesar karena banyak dokumen sama panjang mendapat jatah lebih;
- pembulatan memakai metode *sisa terbesar*: semua kuota dibulatkan ke bawah, lalu sisa sampel dibagikan ke stratum dengan pecahan terbesar, sehingga totalnya persis sama dengan permintaan.

Sampling dilakukan pada indeks, bukan pada list of dict, jadi tidak ada salinan data per stratum.

In [14]:
def length_strata(lengths, num_strata=10):
    # Nomor stratum 0..num_strata-1 untuk setiap dokumen; tidak ada yang tertinggal.
    inner_bounds = np.percentile(lengths, np.linspace(0, 100, num_strata + 1))[1:-1]
    return np.searchsorted(inner_bounds, lengths, side="right")


def proportional_quota(stratum_sizes, num_samples):
    # Kuota sebanding ukuran stratum; sisa pembulatan diberikan ke pecahan terbesar.
    exact = num_samples * stratum_sizes / stratum_sizes.sum()
    quota = np.floor(exact).astype(int)
    leftover = num_samples - quota.sum()
    quota[np.argsort(quota - exact)[:leftover]] += 1
    return quota


def stratified_indices(lengths, num_samples, rng, num_strata=10):
    if num_samples > len(lengths):
        raise ValueError("num_samples melebihi ukuran populasi")
    stratum = length_strata(lengths, num_strata)
    quota = proportional_quota(np.bincount(stratum, minlength=num_strata), num_samples)
    picks = [rng.choice(np.flatnonzero(stratum == s), size=q, replace=False) for s, q in enumerate(quota)]
    return np.sort(np.concatenate(picks))


def stratified_length_sampling(dataset, num_samples, num_strata=10, seed=0):
    lengths = np.array([len(example["text"]) for example in dataset])
    idx = stratified_indices(lengths, num_samples, np.random.default_rng(seed), num_strata)
    return Dataset.from_list([dataset[i] for i in idx])


sampled_dataset = stratified_length_sampling(corpus, NUM_SAMPLES, seed=42)
strata = length_strata(lengths)
quota = proportional_quota(np.bincount(strata, minlength=10), NUM_SAMPLES)

print(sampled_dataset)
print("kuota per stratum:", quota.tolist(), "| total:", int(quota.sum()))
assert len(sampled_dataset) == NUM_SAMPLES
assert strata[lengths.argmax()] == 9          # dokumen terpanjang kini punya stratum
print("dokumen terpanjang: populasi", lengths.max(), "| sampel", max(len(t) for t in sampled_dataset["text"]))

Dataset({
    features: ['id', 'text'],
    num_rows: 305
})
kuota per stratum: [31, 30, 31, 31, 30, 30, 31, 31, 30, 30] | total: 305
dokumen terpanjang: populasi 69815 | sampel 26437


Kini 305 sampel diminta dan 305 didapat. Kuota per stratum hanya berbeda satu dokumen antarstratum, mengikuti ukuran stratum dan sisa pembulatan. Dokumen terpanjang punya peluang terpilih, walaupun dalam satu sampel tertentu ia belum tentu terpilih; itu wajar untuk sampling acak.

### Under the hood: seberapa setia sampel terhadap populasi?

Klaim buku: stratified sampling menjaga distribusi lebih baik daripada random sampling. Kita ukur dengan **statistik Kolmogorov–Smirnov (KS)**. Caranya, gambar kurva kumulatif panjang dokumen untuk sampel dan untuk populasi, lalu ambil jarak vertikal terbesar di antara keduanya. Nilai 0 berarti kedua distribusi identik. Nilai 0,05 berarti di suatu panjang L, proporsi "dokumen lebih pendek dari L" di sampel meleset 5 poin persen dari populasi. Setiap ukuran sampel diulang 20 kali dengan seed berbeda, lalu hasilnya dirata-rata.

In [15]:
from scipy.stats import ks_2samp

print(f"{'n':>5} | {'KS random':>9} | {'KS stratified':>13} | rasio")
for n in (100, 300, 1000):
    ks_random, ks_strat = [], []
    for trial in range(20):
        rng = np.random.default_rng(1000 + trial)
        random_idx = rng.choice(len(lengths), size=n, replace=False)
        strat_idx = stratified_indices(lengths, n, rng)
        ks_random.append(ks_2samp(lengths[random_idx], lengths).statistic)
        ks_strat.append(ks_2samp(lengths[strat_idx], lengths).statistic)
    ratio = np.mean(ks_random) / np.mean(ks_strat)
    print(f"{n:>5} | {np.mean(ks_random):>9.4f} | {np.mean(ks_strat):>13.4f} | {ratio:.2f}x")

    n | KS random | KS stratified | rasio
  100 |    0.0830 |        0.0390 | 2.13x
  300 |    0.0491 |        0.0211 | 2.33x
 1000 |    0.0209 |        0.0112 | 1.86x


Pada run validasi lokal, KS stratified lebih kecil dari KS random di ketiga ukuran sampel, dengan rasio 2,13×, 2,33×, dan 1,86× untuk n = 100, 300, dan 1000. Rasio ini tidak menghilang ketika n membesar: kedua jarak turun bersama, kira-kira sebanding 1/√n. Mekanismenya *variance reduction*. Stratifikasi menghapus satu sumber keacakan, yaitu "stratum mana yang kebetulan terwakili berlebih", sehingga yang tersisa hanya keacakan di dalam setiap stratum.

Bandingkan juga secara diagonal. KS stratified untuk n = 300 (0,0211) sebanding dengan KS random untuk n = 1000 (0,0209). Untuk distribusi panjang, 300 dokumen terstratifikasi sekitar sesetia 1.000 dokumen acak.

Batasnya: stratifikasi hanya menjamin kesetiaan pada atribut yang dipakai untuk membuat strata, di sini panjang teks. Untuk bahasa, topik, atau kualitas, sampel ini tidak dijamin lebih baik daripada sampel acak.

## 4. Distributed processing dengan Dask

Sampling mengurangi jumlah data, tetapi sisanya tetap harus diproses. Jika satu core tidak cukup, pekerjaan dibagi ke banyak core atau mesin. Buku menyebut dua alat. **Apache Spark** kuat untuk batch produksi, tetapi butuh cluster. **Dask** meniru API pandas, sehingga kode Python yang sudah ada bisa diskalakan dengan sedikit perubahan.

Ide Dask: DataFrame besar dipecah menjadi **partisi** (potongan baris). Setiap operasi dicatat sebagai graf tugas dan baru dijalankan saat `.compute()` dipanggil. Scheduler lalu membagikan tugas per partisi ke worker.

> **Listing buku: `distributed_preprocessing` — perlu diperbaiki**
>
> ```python
> def preprocess(text):
>     # Implement your preprocessing logic here
>     return processed_text
>
> df['processed_text'] = df['text'].map(preprocess)
> ```
>
> `processed_text` tidak pernah didefinisikan. Dask menangkap kesalahan ini sebelum komputasi: untuk menebak tipe kolom hasil, `map` langsung menjalankan fungsi pada data contoh. Pada run historis dan validasi lokal, hasilnya `ValueError: Metadata inference failed in map` yang membungkus `NameError`. Versi di bawah memakai fungsi sungguhan dan `meta=` eksplisit (nama dan tipe kolom hasil), sehingga Dask tidak perlu menebak.

In [16]:
import dask.dataframe as dd
from dask.distributed import Client

csv_path = WORK_DIR / "c4_corpus.csv"
pd.DataFrame(corpus).to_csv(csv_path, index=False)
print(f"{csv_path}: {csv_path.stat().st_size / 2**20:.1f} MB, {len(corpus)} baris")

ch04_work/c4_corpus.csv: 6.6 MB, 3000 baris


In [17]:
def preprocess_text(text):
    return " ".join(str(text).lower().split())


def distributed_preprocessing(data_path, num_partitions):
    # blocksize 64 MB: file kecil ini hanya 1 blok. Pada CSV besar berisi teks multi-baris,
    # batas blok bisa membelah satu record; format kolumnar (bagian 6) tidak punya masalah ini.
    df = dd.read_csv(data_path, blocksize="64MB", keep_default_na=False)
    df = df.repartition(npartitions=num_partitions)
    df["processed_text"] = df["text"].map(preprocess_text, meta=("text", "object"))
    return df.compute()


n_workers = max(1, min(2, os.cpu_count() or 1))
with Client(n_workers=n_workers, threads_per_worker=1, processes=True, dashboard_address=None) as client:
    print("worker aktif:", len(client.scheduler_info()["workers"]))
    timings = {}
    for nparts in sorted({1, n_workers, 100}):
        runs = []
        for _ in range(3):
            t0 = time.perf_counter()
            processed_data = distributed_preprocessing(csv_path, nparts)
            runs.append(time.perf_counter() - t0)
        timings[nparts] = float(np.median(runs))

expected = [preprocess_text(d["text"]) for d in corpus]
assert processed_data["processed_text"].tolist() == expected
for nparts, seconds in timings.items():
    print(f"{nparts:>3} partisi: {seconds:.2f} s (median 3 run)")

INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:45987
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:44031'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:43247'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:33873 name: 1
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:33873
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:47404
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:38435 name: 0
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:38435
INFO:distributed.core:Starting established connection to tcp://127

worker aktif: 2


INFO:distributed.scheduler:Remove client Client-740b837f-cf9e-11f1-82f8-0242ac1c000c
INFO:distributed.core:Received 'close-stream' from tcp://127.0.0.1:47414; closing.
INFO:distributed.scheduler:Remove client Client-740b837f-cf9e-11f1-82f8-0242ac1c000c
INFO:distributed.scheduler:Close client connection: Client-740b837f-cf9e-11f1-82f8-0242ac1c000c
INFO:distributed.scheduler:Retire worker addresses (stimulus_id='retire-workers-1792841498.470431') (0, 1)
INFO:distributed.nanny:Closing Nanny at 'tcp://127.0.0.1:44031'. Reason: nanny-close
INFO:distributed.nanny:Nanny asking worker to close. Reason: nanny-close
INFO:distributed.nanny:Closing Nanny at 'tcp://127.0.0.1:43247'. Reason: nanny-close
INFO:distributed.nanny:Nanny asking worker to close. Reason: nanny-close
INFO:distributed.core:Received 'close-stream' from tcp://127.0.0.1:47410; closing.
INFO:distributed.scheduler:Remove worker addr: tcp://127.0.0.1:38435 name: 0 (stimulus_id='handle-worker-cleanup-1792841498.4814572')
INFO:distri

  1 partisi: 0.44 s (median 3 run)
  2 partisi: 0.45 s (median 3 run)
100 partisi: 1.37 s (median 3 run)


Assert memastikan ke-3.000 baris hasil Dask identik dengan versi Python biasa, jadi yang dibandingkan hanya waktunya. Baca ketiga angka itu sebagai biaya pembagian kerja:

- **1 partisi**: hanya satu worker yang bekerja, sementara worker lain menganggur.
- **partisi = jumlah worker**: kerja dibagi rata ke semua worker.
- **100 partisi** untuk file beberapa MB: ratusan tugas kecil, masing-masing dengan ongkos penjadwalan dan pengiriman data.

Pada dua run validasi lokal (Mac, 2 worker proses), 100 partisi konsisten paling lambat: 0,84 s dan 1,22 s. Urutan 1 dan 2 partisi justru tidak konsisten (0,89 vs 0,20 s di run pertama, 0,16 vs 0,18 s di run kedua). Untuk file 6,6 MB, kerja sebenarnya sangat kecil sehingga ongkos tetap dan noise lebih besar daripada manfaat paralelisme; manfaat itu baru terlihat ketika setiap partisi berisi cukup banyak kerja. Angka di Colab akan berbeda, karena waktunya juga mencakup membaca CSV dan mengirim hasil kembali ke notebook. Pedoman Dask adalah partisi berukuran puluhan sampai ratusan MB, bukan sekadar sebanyak mungkin. Di cluster sungguhan, kode yang sama tidak berubah; yang berubah hanya argumen `Client(...)`, yang diarahkan ke scheduler jarak jauh.

## 5. Sharding: membagi data secara deterministik

Dask memutuskan pembagian data sendiri. Pada penyimpanan dan training terdistribusi, kita sering perlu mengatur pembagiannya: data dipecah menjadi **shard** (potongan independen) yang disimpan atau diproses di mesin berbeda. Ada dua syarat. Ukuran shard harus seimbang, dan setiap shard tetap mewakili data keseluruhan, karena shard yang bias memberi gradien yang bias pada data-parallel training.

Strategi yang dibahas buku, diringkas:

| Strategi | Cara menentukan shard | Cocok untuk |
|---|---|---|
| Hash | hash(key) mod N | data tanpa query rentang |
| Range | rentang nilai berurutan | time series, log |
| Geographic | wilayah | query berbasis lokasi |
| Key-value / directory | tabel pemetaan manual atau layanan lookup | mengendalikan hotspot, distribusi yang berubah |
| Consistent hashing | posisi di cincin hash | jumlah shard yang sering berubah |
| Round-robin | bergiliran | sederhana, tanpa query rentang |
| Workload / tag-based | pola akses atau label domain | traffic timpang, multi-tenant |

Listing buku memakai strategi pertama: hash MD5 dari `id`, lalu modulo jumlah shard. Listing ini berjalan apa adanya.

In [18]:
def shard_data(dataset, num_shards):
    shards = [[] for _ in range(num_shards)]
    for item in dataset:
        # Use a hash function to determine the shard
        shard_index = int(hashlib.md5(item["id"].encode()).hexdigest(), 16) % num_shards
        shards[shard_index].append(item)
    return shards


sharded_data = shard_data(corpus, num_shards=10)
sizes = [len(s) for s in sharded_data]
print("ukuran 10 shard:", sizes, "| total:", sum(sizes))

ukuran 10 shard: [316, 300, 290, 295, 324, 297, 280, 311, 260, 327] | total: 3000


Rata-rata idealnya 300 dokumen per shard. Pada run validasi lokal, ukurannya berkisar 260–327. Apakah simpangan sebesar itu wajar akan diukur sebentar lagi. Sifat yang lebih penting dari hash adalah **deterministik**: `id` yang sama selalu jatuh di shard yang sama, di mesin mana pun dan kapan pun dijalankan. Dengan begitu, sistem terdistribusi bisa menemukan data tanpa tabel lookup.

> **Listing buku: `process_with_sharding` — sketsa yang belum lengkap**
>
> ```python
> def process_with_sharding(dataset: List[Dict], num_shards: int) -> List[Dict]:
>     shards = shard_data(dataset, num_shards)
>     with ProcessPoolExecutor(max_workers=num_shards) as executor:
>         processed_shards = list(executor.map(process_shard, shards))
>     # ... hasil digabung ke aggregated_results, tetapi tidak ada return
> ```
>
> Pada run historis (Python 3.12), sel ini gagal di baris `def` dengan `NameError: name 'List' is not defined`. Python 3.14 menunda evaluasi anotasi, sehingga di sana `def` lolos dan error baru muncul saat fungsi dipanggil (validasi lokal: `NameError: name 'ProcessPoolExecutor' is not defined`). Setelah import ditambahkan pun masih ada dua lubang: `process_shard` tidak didefinisikan di bagian mana pun bab ini, dan fungsi tidak punya `return`, sehingga selalu mengembalikan `None`. Versi lengkap ada di bawah.

Versi lengkap menambahkan import, mengisi `process_shard` dengan kerja nyata (menghitung kata per dokumen), mengembalikan hasilnya, dan membatasi jumlah proses ke jumlah CPU. Sama seperti worker `DataLoader` tadi, fungsi yang dijalankan di proses lain disimpan di modul kecil agar bisa di-import oleh proses anak.

In [19]:
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
from typing import Dict, List

(WORK_DIR / "ch04_shard_worker.py").write_text('''
def process_shard(shard):
    return [{"id": item["id"], "n_words": len(item["text"].split())} for item in shard]
''')
from ch04_shard_worker import process_shard


def process_with_sharding(dataset: List[Dict], num_shards: int) -> List[Dict]:
    shards = shard_data(dataset, num_shards)
    max_workers = min(num_shards, os.cpu_count() or 1)
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        processed_shards = list(executor.map(process_shard, shards))
    aggregated_results = []
    for shard_results in processed_shards:
        aggregated_results.extend(shard_results)
    return aggregated_results


word_counts = process_with_sharding(corpus, num_shards=4)

assert Counter(r["id"] for r in word_counts) == Counter(d["id"] for d in corpus)
total_words = sum(r["n_words"] for r in word_counts)
assert total_words == sum(len(d["text"].split()) for d in corpus)
print(f"{len(word_counts)} hasil dari 4 shard | {total_words:,} kata | rata-rata {total_words / len(word_counts):.0f} kata/dokumen")

3000 hasil dari 4 shard | 1,120,855 kata | rata-rata 374 kata/dokumen


Dua assert itu adalah inti pola *map-reduce*. Pertama, setiap `id` muncul tepat sekali di hasil, jadi tidak ada shard yang hilang atau terhitung dua kali. Kedua, total kata sama dengan hasil perhitungan langsung. Urutan hasil mengikuti urutan shard, bukan urutan korpus; karena itu `id` ikut dibawa agar hasilnya bisa digabung kembali ke dokumen asalnya.

### Under the hood: seberapa seimbang, dan berapa harga menambah shard?

Ada dua pertanyaan untuk hash modulo.

1. **Keseimbangan.** Jika MD5 memperlakukan URL seperti bilangan acak, ukuran setiap shard mengikuti distribusi binomial: 3.000 dokumen dengan peluang 1/10 per shard. Simpangan baku teoretisnya √(3000 × 0,1 × 0,9) ≈ 16,4 dokumen (rumus).
2. **Rebalancing.** Jika cluster bertambah dari 10 ke 11 shard, berapa persen dokumen harus pindah? Dengan modulo, sebuah key tetap di tempatnya hanya jika `h mod 10 = h mod 11`. Itu terjadi pada 10 dari setiap 110 nilai hash, jadi sekitar 1 − 1/11 ≈ 90,9% key pindah (rumus). **Consistent hashing** menaruh shard dan key di sebuah cincin hash, dan setiap key menjadi milik shard pertama searah jarum jam. Shard baru hanya merebut sebagian kecil cincin, idealnya 1/11 ≈ 9,1% key. Agar porsi setiap shard di cincin rata, setiap shard diwakili 64 titik virtual.

In [20]:
import bisect

ids = [d["id"] for d in corpus]


def md5_int(key):
    return int(hashlib.md5(key.encode()).hexdigest(), 16)


shard_sizes = np.bincount([md5_int(k) % 10 for k in ids], minlength=10)
expected_sd = np.sqrt(len(ids) * 0.1 * 0.9)
print(f"(a) simpangan baku ukuran shard: terukur {shard_sizes.std():.1f} | teori binomial {expected_sd:.1f}")

moved_mod = np.mean([md5_int(k) % 10 != md5_int(k) % 11 for k in ids])

VNODES = 64


def build_ring(n_shards):
    points = sorted((md5_int(f"shard{s}-vnode{v}"), s) for s in range(n_shards) for v in range(VNODES))
    return [p for p, _ in points], [s for _, s in points]


def ring_shard(key, ring):
    positions, owners = ring
    i = bisect.bisect_left(positions, md5_int(key))
    return owners[i % len(owners)]          # melewati ujung cincin -> kembali ke awal


ring10, ring11 = build_ring(10), build_ring(11)
moved_ring = np.mean([ring_shard(k, ring10) != ring_shard(k, ring11) for k in ids])

print("(b) key yang pindah saat 10 -> 11 shard:")
print(f"    hash modulo       : {moved_mod:6.1%}  (teori {1 - 1/11:.1%})")
print(f"    consistent hashing: {moved_ring:6.1%}  (ideal {1/11:.1%})")

(a) simpangan baku ukuran shard: terukur 19.5 | teori binomial 16.4
(b) key yang pindah saat 10 -> 11 shard:
    hash modulo       :  90.5%  (teori 90.9%)
    consistent hashing:   8.7%  (ideal 9.1%)


**(a)** Pada run validasi lokal, simpangan baku terukur 19,5 dibanding teori 16,4. Simpangan baku yang dihitung dari hanya 10 shard berfluktuasi sendiri sekitar ±24% (rumus distribusi chi-kuadrat dengan 9 derajat kebebasan), jadi angka itu masih berada dalam fluktuasi hash yang ideal. Tidak terlihat struktur URL, seperti domain atau prefix yang sama, yang bocor ke penempatan shard.

**(b)** Pada run validasi lokal, menambah satu shard memindahkan 90,5% key dengan modulo, tetapi hanya 8,7% dengan consistent hashing. Selisihnya sekitar satu orde besaran. Di sistem penyimpanan sungguhan, setiap key yang pindah berarti data yang dikirim lewat jaringan selama rebalancing.

## 6. Format penyimpanan: baris atau kolom?

Data hasil preprocessing perlu disimpan, dan format file menentukan ukurannya di disk serta seberapa cepat data bisa dibaca ulang. Perbedaan dasarnya ada di tata letak:

- **row-based** (CSV, JSON Lines): satu record disimpan utuh, lalu record berikutnya. Untuk membaca satu kolom, seluruh file tetap harus di-parse.
- **columnar** (Parquet, Arrow): semua nilai satu kolom disimpan berdekatan. Satu kolom bisa dibaca tanpa menyentuh kolom lain, dan nilai sejenis yang berdekatan lebih mudah dikompres.

Table 4.1 buku, diringkas:

| Fitur | CSV | JSON | Parquet | Arrow |
|---|---|---|---|---|
| Tata letak | baris | baris | kolom | kolom |
| Kompresi | dasar | buruk | sangat baik | sangat baik |
| Struktur nested dan skema | tidak | terbatas | ya | ya |
| Akses acak | buruk | buruk | baik | sangat baik |
| Use case | dataset kecil | respons API | analitik besar | in-memory |

Untuk NLP, Parquet bisa menyimpan kolom kompleks dalam satu file (Table 4.2 buku): teks, daftar token, vektor embedding, struct metadata, dan skor bahasa.

> **Listing buku: `convert_to_parquet` — perlu diperbaiki**
>
> ```python
> def convert_to_parquet(dataset, output_path):
>     table = pa.Table.from_pydict(dataset[0])
>     pq.write_table(table, output_path)
> ```
>
> `from_pydict` mengharapkan **dict berisi kolom** (`{"id": [semua id], "text": [semua teks]}`), sedangkan `dataset[0]` adalah **satu record**. PyArrow lalu menganggap setiap string sebagai kolom berisi karakter. Pada run historis dan validasi lokal, hasilnya `ArrowInvalid: Column 1 named text expected length 62 but got length 747`, yaitu URL 62 karakter dibandingkan teks 747 karakter. Seandainya panjangnya kebetulan sama, hasilnya adalah tabel karakter dari satu dokumen, dan semua dokumen lain hilang tanpa error.

Perbaikannya memakai `pa.Table.from_pylist`, yang memang menerima list of records. Fungsi baca mengembalikan list of records juga, sehingga roundtrip bisa dicek langsung terhadap korpus.

In [21]:
import pyarrow as pa
import pyarrow.parquet as pq


def convert_to_parquet(dataset, output_path):
    table = pa.Table.from_pylist(dataset)      # list of records -> tabel kolumnar
    pq.write_table(table, output_path)


def read_from_parquet(file_path):
    return pq.read_table(file_path).to_pylist()


parquet_path = WORK_DIR / "large_dataset.parquet"
convert_to_parquet(corpus, parquet_path)
loaded_dataset = read_from_parquet(parquet_path)

assert loaded_dataset == corpus
print("skema:", ", ".join(f"{field.name}: {field.type}" for field in pq.read_schema(parquet_path)))
print(f"ukuran: Parquet {parquet_path.stat().st_size / 2**20:.1f} MB vs CSV {csv_path.stat().st_size / 2**20:.1f} MB (data sama)")

skema: id: string, text: string
ukuran: Parquet 4.0 MB vs CSV 6.6 MB (data sama)


Roundtrip identik untuk semua dokumen. Pada run validasi lokal, Parquet (kompresi Snappy bawaan) berukuran 4,0 MB dibanding CSV 6,6 MB, atau 39% lebih kecil. Penghematannya tidak dramatis karena isi file didominasi teks bebas yang sulit dikompres. Pada kolom numerik atau kategorikal yang nilainya berulang, seperti label, skor, dan timestamp, rasio kompresinya jauh lebih besar.

Sekarang contoh kolom kompleks dari Table 4.2: embedding dari bagian 2 disimpan bersama teks dan panjangnya. Parquet mencatat ukuran setiap kolom di metadata, sehingga kita bisa melihat berapa byte yang perlu dibaca untuk mengambil satu kolom.

In [22]:
emb_table = pa.table({
    "id": [d["id"] for d in corpus[:n_emb]],
    "text": [d["text"] for d in corpus[:n_emb]],
    "n_chars": pa.array(lengths[:n_emb], type=pa.int32()),
    "embedding": pa.FixedSizeListArray.from_arrays(pa.array(doc_emb.reshape(-1)), doc_emb.shape[1]),
})
emb_path = WORK_DIR / "corpus_with_embeddings.parquet"
pq.write_table(emb_table, emb_path)

meta = pq.ParquetFile(emb_path).metadata
col_bytes = Counter()
for rg in range(meta.num_row_groups):
    for j in range(meta.num_columns):
        column = meta.row_group(rg).column(j)
        col_bytes[column.path_in_schema.split(".")[0]] += column.total_compressed_size
total_bytes = sum(col_bytes.values())
for name, nbytes in col_bytes.items():
    print(f"{name:>10}: {nbytes / 2**20:7.3f} MB ({nbytes / total_bytes:6.2%} dari data kolom)")

n_chars_back = pq.read_table(emb_path, columns=["n_chars"]).column("n_chars").to_numpy()
emb_back = pq.read_table(emb_path, columns=["embedding"]).column("embedding").combine_chunks()
emb_back = emb_back.flatten().to_numpy().reshape(n_emb, -1)
assert np.array_equal(n_chars_back, lengths[:n_emb])
assert np.array_equal(emb_back, doc_emb)
print(f"skema embedding: {emb_table.schema.field('embedding').type}")

        id:   0.150 MB ( 1.12% dari data kolom)
      text:   3.860 MB (28.79% dari data kolom)
   n_chars:   0.012 MB ( 0.09% dari data kolom)
 embedding:   9.384 MB (70.00% dari data kolom)
skema embedding: fixed_size_list<item: float>[768]


Kolom `n_chars` hanya menempati sebagian sangat kecil dari file, sedangkan `text` dan `embedding` mengambil hampir seluruhnya. Karena itu query seperti "pilih dokumen yang panjangnya di atas X" cukup membaca porsi kecil tersebut tanpa menyentuh teks atau embedding. Inilah keunggulan format kolumnar pada pipeline LLM: filter berdasarkan metadata dulu, baru baca kolom besar untuk baris yang lolos. Embedding juga kembali bit-per-bit identik, jadi menyimpannya sebagai `fixed_size_list<float>` tidak mengubah nilainya.

### Under the hood: empat format pada data yang sama

Ukuran file baru separuh cerita. Kita tulis tabel yang sama ke CSV, JSON Lines, Parquet, dan Arrow/Feather, lalu mengukur waktu tulis, baca penuh, dan baca satu kolom. Tabelnya berisi id, teks, panjang, dan satu kolom `language_score` **sintetis** berseed sebagai contoh metadata numerik.

Supaya perbandingannya adil:

- setiap operasi baca menghasilkan DataFrame pandas;
- setiap angka adalah median dari `TIMING_REPEATS` ulangan;
- semua file baru saja ditulis, jadi sudah ada di cache memori sistem operasi.

Poin ketiga berarti yang terukur adalah biaya parse dan decode, bukan kecepatan disk. Pada disk dingin atau penyimpanan jaringan, selisih jumlah byte yang dibaca menjadi lebih menentukan.

In [23]:
import pyarrow.feather as feather

bench = pd.DataFrame({
    "id": [d["id"] for d in corpus],
    "text": [d["text"] for d in corpus],
    "n_chars": lengths,
    "language_score": np.random.default_rng(0).uniform(0.7, 1.0, len(corpus)).round(4),   # sintetis
})


def median_seconds(fn, repeats=TIMING_REPEATS):
    runs = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        runs.append(time.perf_counter() - t0)
    return float(np.median(runs))


formats = {
    "CSV": ("csv",
            lambda p: bench.to_csv(p, index=False),
            lambda p: pd.read_csv(p),
            lambda p: pd.read_csv(p, usecols=["language_score"])),
    "JSONL": ("jsonl",
              lambda p: bench.to_json(p, orient="records", lines=True, force_ascii=False),
              lambda p: pd.read_json(p, lines=True),
              lambda p: pd.read_json(p, lines=True)[["language_score"]]),
    "Parquet": ("parquet",
                lambda p: bench.to_parquet(p),
                lambda p: pd.read_parquet(p),
                lambda p: pd.read_parquet(p, columns=["language_score"])),
    "Arrow": ("arrow",
              lambda p: feather.write_feather(bench, p),
              lambda p: feather.read_feather(p),
              lambda p: feather.read_feather(p, columns=["language_score"])),
}

print(f"{'format':>8} | {'ukuran':>8} | {'tulis':>8} | {'baca penuh':>10} | {'baca 1 kolom':>12}")
for name, (ext, write, read_full, read_col) in formats.items():
    path = WORK_DIR / f"bench.{ext}"
    t_write = median_seconds(lambda: write(path))
    t_full = median_seconds(lambda: read_full(path))
    t_col = median_seconds(lambda: read_col(path))
    assert len(read_full(path)) == len(bench)
    assert list(read_col(path).columns) == ["language_score"]
    print(f"{name:>8} | {path.stat().st_size / 2**20:6.1f}MB | {t_write * 1e3:6.0f}ms | "
          f"{t_full * 1e3:8.0f}ms | {t_col * 1e3:10.0f}ms")

  format |   ukuran |    tulis | baca penuh | baca 1 kolom
     CSV |    6.6MB |    219ms |       74ms |         40ms
   JSONL |    6.8MB |    147ms |      114ms |        110ms
 Parquet |    4.0MB |     45ms |       31ms |          2ms
   Arrow |    4.2MB |     32ms |       13ms |          1ms


Cara membaca tabelnya:

- **Ukuran**: format kolumnar yang terkompresi biasanya lebih kecil daripada format baris. Pada run validasi lokal: CSV 6,6 MB, JSONL 6,8 MB, Parquet 4,0 MB, Arrow 4,2 MB.
- **Baca 1 kolom** adalah pembeda utamanya. CSV dan JSONL tetap harus membaca setiap baris untuk mengambil satu kolom (JSONL bahkan mem-parse seluruh record), sedangkan Parquet dan Arrow melompat langsung ke potongan kolom itu. Selisihnya makin besar ketika kolom yang dilewati makin besar; di sini kolom yang dilewati adalah teks, yang memakan hampir seluruh file.
- **Baca penuh**: Arrow menyimpan data dengan tata letak yang hampir sama dengan di memori, jadi biaya decode-nya paling kecil. Selisih format lain di baca penuh tidak sebesar di baca satu kolom, karena semua data tetap harus dimuat. Pada run validasi lokal, CSV dan Parquet bahkan hampir sama cepat, dan JSONL paling lambat.

Waktunya bergantung pada CPU dan versi library, jadi bandingkan urutan dan rasio antarformat di output-mu sendiri, bukan nilai milidetiknya. Pada 3.000 baris, selisih absolutnya hanya milidetik.

## 7. Streaming untuk continuous training

Sejauh ini korpusnya tetap: datanya sudah ada dan ukurannya diketahui. Di banyak sistem, data datang terus-menerus, seperti interaksi pengguna, log, dan dokumen baru, sementara model diperbarui secara berkala. Arsitektur standarnya terdiri dari dua bagian:

- **Apache Kafka** sebagai tulang punggung: *producer* menulis pesan ke sebuah **topic**, lalu *consumer* membacanya. Beberapa broker memberi redundansi dan throughput.
- **Faust**, library Python di atas `asyncio`, untuk menulis consumer sebagai fungsi asinkron yang memproses event satu per satu.

> **Listing buku: consumer Faust — tidak dijalankan di notebook ini**
>
> ```python
> app = faust.App('llm-training', broker='kafka://localhost:9092')
> topic = app.topic('raw-text', value_type=Text)
>
> @app.agent(topic)
> async def process(stream):
>     async for text in stream:
>         processed_text = preprocess(text.content)
> ```
>
> Kode ini butuh broker Kafka yang hidup di `localhost:9092`. Selain itu, `app.main()` menjalankan worker yang terus menunggu event, sehingga sel notebook tidak pernah selesai. `preprocess` juga tidak didefinisikan di listing. Proyek `faust` asli dari Robinhood sudah tidak aktif dikembangkan; komunitas melanjutkannya sebagai `faust-streaming`. Kode ini tidak diuji di revisi ini.

Ide pentingnya tetap bisa dicoba tanpa Kafka. Simulasi di bawah memakai `asyncio.Queue` sebagai pengganti topic:

- **producer** mengirim 1.000 dokumen;
- **consumer** menormalisasi teks lalu mengumpulkannya menjadi **micro-batch** berisi 96 dokumen, yaitu unit yang akan diberikan ke satu langkah training;
- antrean dibatasi 64 pesan. Jika consumer lambat, producer otomatis menunggu. Mekanisme ini disebut *backpressure*, dan mencegah memori meledak.

In [24]:
import asyncio

STREAM_DOCS = min(1000, len(corpus))
MICRO_BATCH = 96
END_OF_STREAM = None      # Kafka tidak punya sinyal akhir; penanda ini hanya agar demo selesai


async def producer(queue, docs):
    for doc in docs:
        await queue.put(doc["text"])
    await queue.put(END_OF_STREAM)


async def training_agent(queue, micro_batch_size):
    batch, batch_sizes = [], []
    while True:
        text = await queue.get()
        if text is END_OF_STREAM:
            break
        batch.append(preprocess_text(text))
        if len(batch) == micro_batch_size:
            batch_sizes.append(len(batch))      # di sistem nyata: train_step(batch)
            batch = []
    if batch:                                   # flush sisa batch; tanpa ini dokumen terakhir hilang
        batch_sizes.append(len(batch))
    return batch_sizes


queue = asyncio.Queue(maxsize=64)
_, batch_sizes = await asyncio.gather(producer(queue, corpus[:STREAM_DOCS]), training_agent(queue, MICRO_BATCH))

assert sum(batch_sizes) == STREAM_DOCS
print(f"{len(batch_sizes)} micro-batch | ukuran: {batch_sizes}")

11 micro-batch | ukuran: [96, 96, 96, 96, 96, 96, 96, 96, 96, 96, 40]


Sebanyak 1.000 dokumen menjadi 10 micro-batch penuh (960 dokumen) ditambah satu batch sisa berisi 40 (hitungan pasti: 1.000 = 10 × 96 + 40). Tanpa blok *flush*, 40 dokumen terakhir tidak pernah sampai ke training, dan bug ini tidak memunculkan error. Di Kafka sungguhan, stream tidak pernah selesai, jadi flush biasanya dipicu oleh batas waktu. Posisi baca (*offset*) juga baru di-commit setelah batch berhasil diproses, supaya data tidak hilang ketika consumer di-restart.

## 8. Loading hemat memori: memmap dan chunking

Bagian terakhir menjawab pertanyaan: bagaimana membaca file yang lebih besar dari RAM? Ada dua pola, dipilih berdasarkan cara data diakses.

- **Memory mapping (memmap)**: file dipetakan ke ruang alamat proses. Array terasa seperti ada di memori, tetapi sistem operasi baru membaca sebuah **halaman** (blok 4 KB di Linux; 16 KB di Mac Apple Silicon) dari disk ketika bagian itu disentuh. Pola ini cocok untuk akses **acak**, misalnya mengambil embedding dokumen tertentu.
- **Chunking**: file dibaca per blok berukuran tetap secara berurutan. Pola ini cocok untuk akses **sekuensial**, misalnya menghitung statistik seluruh log.

Listing memmap buku berjalan apa adanya. Di sini listing itu dipakai untuk embedding asli dari bagian 2.

In [25]:
def create_memmap_dataset(dataset, output_file):
    # Determine the shape of the dataset
    num_samples = len(dataset)
    sample_shape = dataset[0]["input"].shape
    # Create a memory-mapped array
    mmap = np.memmap(output_file, dtype="float32", mode="w+", shape=(num_samples, *sample_shape))
    # Write data to the memory-mapped array
    for i, sample in enumerate(dataset):
        mmap[i] = sample["input"]
    # Flush to disk
    mmap.flush()


def load_memmap_dataset(file_path, shape):
    # Load the memory-mapped array
    return np.memmap(file_path, dtype="float32", mode="r", shape=shape)


emb_dataset = [{"id": corpus[i]["id"], "input": doc_emb[i]} for i in range(n_emb)]
mmap_path = WORK_DIR / "embeddings.mmap"
create_memmap_dataset(emb_dataset, mmap_path)
mmap_dataset = load_memmap_dataset(mmap_path, shape=(n_emb, doc_emb.shape[1]))

assert np.array_equal(mmap_dataset, doc_emb)
print(f"file: {mmap_path.stat().st_size / 2**20:.2f} MB = {n_emb} x {doc_emb.shape[1]} x 4 byte | "
      f"objek: {type(mmap_dataset).__name__} {mmap_dataset.shape} {mmap_dataset.dtype}")

file: 8.79 MB = 3000 x 768 x 4 byte | objek: memmap (3000, 768) float32


Ukuran file mentah ini tepat `n × dimensi × 4 byte`, tanpa header apa pun. Membukanya dengan `mode="r"` tidak membaca data; sebuah baris baru dimuat ketika disentuh. Kelemahannya, shape dan dtype tidak tersimpan di file, sehingga pemanggil harus mengingatnya sendiri. Format `.npy` menyimpan keduanya di header kecil dan tetap bisa di-memmap (`np.load(..., mmap_mode="r")`). Format itulah yang dipakai pada pengukuran di akhir bagian ini.

Listing chunking buku juga berjalan apa adanya. Agar pelajarannya terlihat, isi kolom `value` dibuat **sintetis** dengan tren naik: dari sekitar 50 di baris awal ke sekitar 150 di baris akhir, seperti metrik harian yang terus tumbuh, ditambah noise. Rata-rata seluruh file menjadi sekitar 100. Mean seluruh file juga dihitung sekali dengan membaca penuh, sebagai kunci jawaban.

In [26]:
n_rows = 200_000
rng = np.random.default_rng(11)
values_csv = WORK_DIR / "large_dataset.csv"
pd.DataFrame({"value": (np.linspace(50, 150, n_rows) + rng.normal(0, 15, n_rows)).round(3)}).to_csv(values_csv, index=False)
true_mean = pd.read_csv(values_csv)["value"].mean()
print(f"{values_csv.name}: {n_rows:,} baris | mean seluruh file = {true_mean:.3f}")

large_dataset.csv: 200,000 baris | mean seluruh file = 99.949


In [27]:
def process_chunk(chunk):
    # Placeholder: process or transform the chunk here
    # For example, compute the mean of a column
    return chunk["value"].mean()


def process_large_csv(file_path, chunk_size=10000):
    results = []
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        result = process_chunk(chunk)
        results.append(result)
    return results


for chunk_size in (10_000, 30_000):
    chunk_means = process_large_csv(values_csv, chunk_size=chunk_size)
    print(f"chunk {chunk_size:>6,}: {len(chunk_means)} chunk | mean dari mean-chunk = {np.mean(chunk_means):.3f} "
          f"(selisih {np.mean(chunk_means) - true_mean:+.3f})")

chunk 10,000: 20 chunk | mean dari mean-chunk = 99.949 (selisih +0.000)
chunk 30,000: 7 chunk | mean dari mean-chunk = 102.085 (selisih +2.136)


Dengan chunk 10.000, 200.000 baris terbagi rata menjadi 20 chunk, sehingga rata-rata dari mean-chunk sama dengan mean file. Dengan chunk 30.000, chunk ketujuh hanya berisi 20.000 baris dari ujung file yang nilainya paling tinggi, tetapi bobotnya disamakan dengan chunk penuh. Hasilnya bergeser sekitar +2 (run validasi lokal: +2,136). Dari tren saja, rata-rata ketujuh mean-chunk adalah (57,5 + 72,5 + 87,5 + 102,5 + 117,5 + 132,5 + 145) / 7 ≈ 102,1 (rumus). Bug ini tidak memunculkan error, dan nilainya tetap terlihat masuk akal.

Perbaikannya: setiap chunk mengembalikan **jumlah dan banyaknya baris**, lalu keduanya digabung di akhir. Pola yang sama berlaku untuk statistik training data apa pun yang dihitung per potongan.

In [28]:
def process_chunk_sum_count(chunk):
    return chunk["value"].sum(), len(chunk)


def streaming_mean(file_path, chunk_size):
    total, count = 0.0, 0
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        chunk_sum, chunk_count = process_chunk_sum_count(chunk)
        total += chunk_sum
        count += chunk_count
    return total / count


exact = streaming_mean(values_csv, chunk_size=30_000)
assert abs(exact - true_mean) < 1e-9
print(f"(sum, count) per chunk 30.000: mean = {exact:.3f} | selisih dengan mean file {exact - true_mean:+.1e}")

(sum, count) per chunk 30.000: mean = 99.949 | selisih dengan mean file -1.4e-14


### Under the hood: memmap vs full load, diukur

Klaim buku: memmap memungkinkan kita bekerja dengan data yang lebih besar dari RAM. Kita uji pada array `.npy` sebesar `MEMMAP_TEST_MB` dengan mengukur **RSS** (*resident set size*, memori fisik yang sedang dipakai proses) setelah tiga langkah: membaca penuh, membuka sebagai memmap, dan mengakses 1.000 baris acak lewat memmap.

Ada satu jebakan pengukuran. RSS menghitung dua jenis halaman sekaligus:

- Halaman **anonim** adalah milik proses, misalnya hasil `np.load` penuh. Halaman ini harus di-swap bila RAM penuh.
- Halaman **file-backed** adalah salinan isi file. Kernel boleh membuangnya kapan saja karena isinya bisa dibaca ulang dari disk.

Di Linux, termasuk Colab, `psutil` memberi kolom `shared` yang kira-kira sama dengan bagian file-backed, sehingga keduanya bisa dipisah. Di sistem lain kolom itu tidak ada dan ditampilkan sebagai `nan`.

Varian terakhir memberi petunjuk `MADV_RANDOM` ke kernel ("akses saya acak, jangan baca halaman tetangga"). Tujuannya melihat apakah kernel ikut memetakan halaman di sekitar baris yang disentuh. File baru saja ditulis dan sudah ada di cache sistem operasi, jadi latensi di bawah adalah latensi cache hangat, bukan latensi disk.

In [29]:
import mmap as mmap_module

rows = MEMMAP_TEST_MB * 2**20 // (256 * 4)
big_path = WORK_DIR / "big.npy"
big = np.lib.format.open_memmap(big_path, mode="w+", dtype="float32", shape=(rows, 256))
for start in range(0, rows, 20_000):
    stop = min(start + 20_000, rows)
    big[start:stop] = np.random.default_rng(start).standard_normal((stop - start, 256), dtype=np.float32)
big.flush()
del big
gc.collect()

proc = psutil.Process()
HAS_SHARED = hasattr(proc.memory_info(), "shared")      # True di Linux


def memory_mb():
    info = proc.memory_info()
    file_backed = info.shared / 2**20 if HAS_SHARED else float("nan")
    return info.rss / 2**20, file_backed


def delta(before, after):
    return tuple(a - b for a, b in zip(after, before))


random_rows = np.random.default_rng(3).integers(0, rows, 1000)


def per_row_us(array):
    t0 = time.perf_counter()
    for r in random_rows:
        float(array[r].sum())
    return (time.perf_counter() - t0) / len(random_rows) * 1e6


results = {}

before = memory_mb()
full = np.load(big_path)
results["full load"] = delta(before, memory_mb())
lat_full = per_row_us(full)
del full
gc.collect()

before = memory_mb()
mm = np.load(big_path, mmap_mode="r")
results["memmap: buka"] = delta(before, memory_mb())
lat_mm = per_row_us(mm)
results["memmap: +1000 akses acak"] = delta(before, memory_mb())
data_offset = mm.offset
del mm
gc.collect()

if hasattr(mmap_module, "MADV_RANDOM"):
    with open(big_path, "rb") as f:
        raw = mmap_module.mmap(f.fileno(), 0, access=mmap_module.ACCESS_READ)
    raw.madvise(mmap_module.MADV_RANDOM)
    before = memory_mb()
    arr = np.frombuffer(raw, dtype=np.float32, offset=data_offset).reshape(rows, 256)
    per_row_us(arr)
    results["memmap + MADV_RANDOM: +1000 akses"] = delta(before, memory_mb())
    del arr
    raw.close()

print(f"file: {big_path.stat().st_size / 2**20:.0f} MB | {rows:,} baris x 256 float32 | platform: {platform.system()}")
print(f"{'langkah':<36} | {'Δ RSS':>8} | {'Δ file-backed':>13} | {'Δ anonim':>9}")
for step, (d_rss, d_file) in results.items():
    print(f"{step:<36} | {d_rss:6.1f}MB | {d_file:11.1f}MB | {d_rss - d_file:7.1f}MB")
print(f"latensi per baris acak (cache hangat): full {lat_full:.1f} µs | memmap {lat_mm:.1f} µs")
if not HAS_SHARED:
    print("catatan: platform ini tidak memisahkan halaman file-backed, dan Δ RSS di luar Linux bisa bias "
          "oleh memori yang dilepas sel sebelumnya; jangan tarik kesimpulan anonim vs file-backed dari tabel ini.")
big_path.unlink()

file: 200 MB | 204,800 baris x 256 float32 | platform: Linux
langkah                              |    Δ RSS | Δ file-backed |  Δ anonim
full load                            |  200.0MB |         0.0MB |   200.0MB
memmap: buka                         |    0.0MB |         0.0MB |     0.0MB
memmap: +1000 akses acak             |   55.2MB |        55.2MB |     0.0MB
memmap + MADV_RANDOM: +1000 akses    |   55.2MB |        55.2MB |     0.0MB
latensi per baris acak (cache hangat): full 2.2 µs | memmap 9.0 µs


Cara membaca tabelnya:

- **full load**: Δ RSS kira-kira sebesar file, dan seluruhnya **anonim**. Memori itu milik proses. Pada file yang lebih besar dari RAM, langkah ini gagal.
- **memmap: buka**: Δ RSS mendekati nol. Membuka file hanya memetakan alamat dan tidak membaca data.
- **memmap: +1000 akses acak**: 1.000 baris × 1 KB idealnya hanya menyentuh sekitar 1.000 halaman (≈ 4 MB di Linux). Jika Δ file-backed jauh lebih besar, kernel ikut memetakan halaman tetangga. Bagian anonimnya tetap mendekati nol, dan itulah alasan memmap aman untuk data yang lebih besar dari RAM: halaman file-backed bisa dibuang kernel kapan saja tanpa swap.
- **MADV_RANDOM**: jika baris ini jauh lebih kecil daripada baris sebelumnya, halaman tambahan tadi berasal dari pembacaan halaman tetangga yang dimatikan oleh petunjuk itu. Jika hampir sama, penyebabnya mekanisme lain. Notebook asli menyimpulkan *readahead* tanpa menguji varian ini; output-mu yang menentukan.

Latensi per baris memmap biasanya lebih tinggi daripada full load, karena setiap halaman yang pertama kali disentuh memicu *page fault*. Itulah harga yang dibayar memmap untuk waktu buka yang hampir nol. Pada disk dingin, selisihnya jauh lebih besar daripada yang terlihat di cache hangat ini.

Bagian ini dirancang untuk Linux (Colab) dan belum diuji di sana. Run validasi lokal di macOS menunjukkan kenapa pemisahan anonim/file-backed penting. Di macOS kolom file-backed tidak tersedia, dan Δ RSS full load untuk array 64 MB berubah-ubah antar-run: −6,3 MB pada run pertama, +51,6 MB pada run kedua. Memori yang dilepas sel-sel sebelumnya baru dikembalikan ke sistem belakangan, sehingga baseline bergeser. Uji terpisah di proses baru memberi +61 MB, sesuai harapan. Di Mac yang sama, 1.000 akses acak konsisten menambah 14,7 MB, kira-kira 1.000 halaman × 16 KB, dan `MADV_RANDOM` tidak mengubahnya (14,5 MB). Jadi di macOS kernel tidak terlihat memetakan halaman tetangga. Apakah Linux berperilaku sama, output Colab-mu yang menjawab.

## Ringkasan

Satu korpus nyata, 3.000 dokumen C4, dibawa melewati seluruh pipeline: aliran → tensor → sampel → partisi → shard → file → micro-batch → memmap. Yang perlu dibawa pulang:

- **Streaming** membaca korpus raksasa per record dari satu shard. `num_proc` hanya berlaku untuk dataset yang sudah dimaterialisasi, bukan untuk aliran.
- **Tokenisasi itu kerja CPU; embedding itu kerja GPU.** Ukur porsi padding serta waktu tokenisasi versus forward pass untuk tahu apakah GPU menunggu data. Pada data ini, bucketing memangkas padding dari 43,9% ke 2,5% (run validasi lokal, tokenizer asli). `num_workers` pada `IterableDataset` bisa menggandakan data jika aliran tidak dibagi per worker.
- **Stratified sampling** butuh batas strata yang mencakup semua dokumen dan kuota yang totalnya tepat. Statistik KS mengukur seberapa setia sampel terhadap populasi, tetapi hanya untuk atribut yang distratifikasi.
- **Dask** menyebar kerja per partisi. Jumlah partisi harus disesuaikan dengan ukuran data dan jumlah worker, dan `meta=` membuat tipe hasil eksplisit.
- **Hash sharding** deterministik dan seimbang (simpangan baku teoretis 16,4 untuk 3.000 dokumen di 10 shard). Namun dengan modulo, menambah satu shard memindahkan sekitar 90,9% key, sedangkan consistent hashing hanya sekitar 9,1% (rumus).
- **Format kolumnar** memungkinkan membaca satu kolom tanpa menyentuh teks, dan metadata Parquet menunjukkan berapa byte yang benar-benar perlu dibaca.
- **Continuous training** bertumpu pada micro-batch, backpressure, dan flush sisa batch.
- **Memmap** membuat data besar terasa ada di memori tanpa memilikinya: halaman file-backed, bukan anonim. **Chunking** harus menggabungkan `(sum, count)`, bukan rata-rata dari rata-rata.

File kerja tersimpan di folder `ch04_work/` (puluhan MB pada profil GPU) dan boleh dihapus setelah selesai.

**Berikutnya:** data yang bisa diproses dalam skala besar juga perlu bisa dilacak. Bab 5 membahas **data versioning** untuk pengembangan LLM.